CS336的Assignment 5一共安排了两个维度的SFT实验，它们共用一套SFT的基础设施（数据集类、训练循环），但目标完全不同。具体来说，CS336的作业设计将SFT分为两部分：第一部分是在数学推理轨迹上微调Qwen2.5-Math-1.5B以提升逐步解题能力，第二部分则是在UltraChat-200K与安全数据上微调模型以实现通用的指令遵循与安全性。

第一部分"推理SFT"是利用更强模型（如DeepSeek R1）生成的推理轨迹作为监督信号，去微调Qwen2.5-1.5B-MATH基座模型。而现在的第二部分"指令SFT"或"对齐SFT"目标不再是如何解数学题，而是如何把模型训练成一个会聊天、会遵循指令、能拒绝危险请求的助手。

## 7.1 指令微调数据与训练
### 7.1.1 指令微调数据集
#### UltraChat
UltraChat 论文定位自己为一个大规模、高质量、信息丰富的指令对话数据集；Hugging Face 的 `ultrachat_200k` 数据卡则给出了后续精炼版本的拆分，其中 `train_sft` 约 208k 条、`test_sft` 约 23.1k 条。

它不是在教模型某一门绝活，而是在把“像一个什么都能聊一点的助手”这件事写进参数里。
#### Safety
来自“Safety-Tuned LLaMAs”论文，关键发现是即使仅用3%的安全样本（几百条演示）就能显著提升模型安全性。它的作用是为模型安装“刹车系统”，学会对危险指令说“不”，但需警惕“过度拒绝”（over-refusal）。

### 7.1.2 Packing
指令微调的数据集有一个显著特征：样本长度极不均匀。一段对话可能只有几十个 token，另一段却长达上千。如果使用传统的 padding 方案，设定 seq_length=2048 后，平均长度 200 token 的样本会让计算量中超过 90% 耗费在无意义的填充符上。这不仅降低吞吐，也让反向传播的梯度中混杂了大量对 pad 位置的无用更新。

Packing 的核心思想是将多条样本拼接成一条连续的 token 长链，首尾以 EOS 分隔，再把这条长链切成等长的训练块。这样，每一个进入 Transformer 的 token 都是真实语料，训练效率理论上可以接近 100%。

具体来说，Packing 的优势体现在以下几个维度：

- 消除填充浪费：每个 token 都是真实数据，不再有填充符号参与计算，token 利用率从 <30% 跃升至 ~100%。
- 提升吞吐量：每个批次包含了更多有效信息，Hugging Face 官方实验报告在指令微调场景下吞吐量最高可提升至原来的约两倍；Unsloth 则宣称结合其内核优化后，Packing 可带来最高约 3 倍的训练吞吐量提升。
- 加速收敛：由于每一步参数更新都基于更多的有效 token（而非填充符号），有效梯度更新频率更高，训练收敛速度相应加快。
```
doc1 tokens + [eos] + doc2 tokens + [eos] + doc3 tokens + [eos] + ...
        ↓
一维长链 all_token_ids / all_label_ids
        ↓
reshape 成 (num_chunks, seq_len + 1)
        ↓
input_ids = chunk[:, :-1]
labels    = chunk[:, 1:]
        ↓
丢掉 labels 全是 -100 的死块
```
#### 实现方式
##### 朴素拼接：基于 EOS 的软分隔
将所有样本拼接为一根长链，再按固定长度切割，同时依赖 EOS token 作为样本之间的逻辑分隔符。

具体流程如下：首先将所有样本拼接为一根长链，如 `doc1 <eos> doc2 <eos> doc3 <eos> ...`；然后将一维长链 reshape 成 `(num_chunks, seq_length + 1)` 的矩阵，并执行自回归偏移（shift）——`input_ids` 取 `[:, :-1]`，`labels` 取 `[:, 1:]`，使每个位置的输入标签精确对应长链中 t+1 位置的 token；最后过滤掉 `labels` 全为 -100 的"死块"。

考虑一个 chunk 中包含两个样本：

- 样本 A（前 500 token）："请详细解释量子力学的基本原理，包括波粒二象性、不确定性原理和量子纠缠..."
- 样本 B（后 500 token）："用一句话总结《红楼梦》的主要内容。"
在朴素拼接下，模型在处理样本 A 的末尾 token 时，其自注意力机制已经可以"看到"样本 B 的开头部分。这会造成两个问题：其一，模型可能提前捕捉到样本 B 的信息，导致对样本 A 的预测被无关上下文干扰；其二，更微妙的是，模型可能会学到一种虚假模式——某些问题后面往往跟着另一些类型的问题——而这种统计关联在真实推理场景中并不成立。

也就是说，局限在于 EOS 只是一个 token，而不是一个注意力屏障。在自注意力机制下，同一个 chunk 内的两个不同样本之间仍然可以互相"看见"，即存在跨样本注意力（cross-example attention）。
##### 边界感知打包（Boundary-Aware Packing）
针对朴素拼接中跨样本注意力的潜在问题，Hugging Face 在 2024 年 8 月推出了正式的边界感知打包方案，配套引入新的 DataCollatorWithFlattening。该方案利用 flash_attn_varlen_func 在计算注意力时显式传入每个样本的实际长度，使每个 token 只能关注到自身样本范围内的其他 token，从而在注意力计算层面严格隔离不同样本。


In [ ]:
import os
import json
import gzip
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb

from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

## 7.2 实验代码

In [ ]:
CFG = {
    # 路径
    "train_path": "data/sft/train.jsonl",
    "eval_path": "data/sft/eval.jsonl",   # 没有可设为 None
    "model_path": "model/Qwen2.5-3B-Base",
    "output_dir": "result/checkpoints/sft_instruct",

    # 训练参数
    "batch_size": 32,          # 逻辑 batch size
    "micro_batch_size": 2,     # 物理 batch size
    "lr": 2e-5,
    "max_seq_len": 2048,
    "epochs": 1,
    "warmup_ratio": 0.03,

    # 监控 / 保存
    "eval_every_steps": 100,
    "save_every_steps": 200,
    "wandb_project": "cs336-sft-instruct",
    "wandb_run_name": "qwen25-3b-ultrachat-safety",

    # 数据集设置
    "shuffle_docs": True,
    "apply_masking": True,
    "seed": 42,
}

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

### 7.2.1 读取 JSONL 数据

In [ ]:
def load_instruction_docs(dataset_path: str) -> List[Dict]:
    """
    读取 jsonl / jsonl.gz 数据，返回原始样本列表。
    """
    path = str(dataset_path)
    docs = []

    open_fn = gzip.open if path.endswith(".gz") else open
    with open_fn(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                docs.append(json.loads(line))

    return docs

### 7.2.2 统一 Alpaca 模板格式
**Alpaca格式**是**指令微调**（Instruction Tuning）领域最通用的数据格式。它的核心结构是一个包含三个字段的JSON对象：

| 字段 | 是否必填 | 作用 |
| :--- | :--- | :--- |
| `instruction` | **必填** | 规定模型需执行的具体任务，如“将以下句子翻译成英文” |
| `input` | 选填 | 为任务提供补充的上下文信息，如待翻译的文本 |
| `output` | **必填** | 模型应给出的标准答案或期望回复 |

一个典型的Alpaca格式数据条目如下：

```json
{
  "instruction": "将以下句子翻译成英文。",
  "input": "今天天气真好，适合出去散步。",
  "output": "The weather is so nice today, perfect for a walk."
}
```
在训练和推理时，需要通过**提示模板**（Prompt Template）将上述结构化字段拼接成一段自然的自然语言指令。Alpaca格式的核心优势在于**简洁明确**，极大地简化了模型对指令的理解与响应，非常适合任务型对话、问答系统等单轮指令场景。

它的主要局限则在于**对多轮对话的支持较弱**。标准的Alpaca格式是为“一指令一回复”的单轮交互设计的。虽然社区格式（如LLaMA-Factory）可通过添加可选的 `history` 字段来支持多轮对话，但处理起来相对复杂。

In [ ]:
def format_instruction_example(prompt: str, response: str) -> str:
    template = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{prompt}\n\n"
        "### Response:\n{response}"
    )
    return template.format(prompt=prompt, response=response)

### 7.2.3 对单条样本做“完整文本编码 + prompt masking”
- full_tokens：完整输入
- labels：Prompt 部分打成 -100，只让 Response 参与 loss

In [ ]:
def tokenize_instruction_example(
    doc: Dict,
    tokenizer,
    apply_masking: bool = True,
) -> Tuple[List[int], List[int]]:
    """
    对单条样本编码，并构造 labels。
    返回:
      full_tokens, labels
    """
    prompt = doc.get("prompt", doc.get("instruction", ""))
    response = doc.get("response", doc.get("output", ""))

    full_text = format_instruction_example(prompt, response)
    full_tokens = tokenizer.encode(full_text, add_special_tokens=False)

    labels = list(full_tokens)

    if apply_masking:
        # 只编码 prompt 模版部分，用来定位要 mask 的前缀
        prompt_only_text = format_instruction_example(prompt, "")
        prompt_only_tokens = tokenizer.encode(prompt_only_text, add_special_tokens=False)

        # 这里不用粗暴地 prompt_tokens[:-1]
        # 更稳的方法：取 full_tokens 和 prompt_only_tokens 的最长公共前缀
        prefix_len = 0
        max_check_len = min(len(full_tokens), len(prompt_only_tokens))
        while prefix_len < max_check_len and full_tokens[prefix_len] == prompt_only_tokens[prefix_len]:
            prefix_len += 1

        # 将 prompt 区域打成 -100
        labels[:prefix_len] = [-100] * prefix_len

    return full_tokens, labels

### 7.2.4 Packing
Packing 的第一步：把所有样本压成一维长链

In [ ]:
def build_packed_token_stream(
    docs: List[Dict],
    tokenizer,
    shuffle_docs: bool = True,
    apply_masking: bool = True,
    seed: int = 42,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    将所有样本构造成 packed 长链:
      all_token_ids
      all_label_ids
    """
    docs = list(docs)

    if shuffle_docs:
        rng = random.Random(seed)
        rng.shuffle(docs)

    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None:
        raise ValueError("Tokenizer 必须有 eos_token_id")

    all_token_ids = []
    all_label_ids = []

    for doc in docs:
        full_tokens, labels = tokenize_instruction_example(
            doc=doc,
            tokenizer=tokenizer,
            apply_masking=apply_masking,
        )

        all_token_ids.extend(full_tokens)
        all_token_ids.append(eos_token_id)

        all_label_ids.extend(labels)
        # EOS 也是模型要学会输出的，所以不能设成 -100
        all_label_ids.append(eos_token_id)

    all_token_ids = torch.tensor(all_token_ids, dtype=torch.long)
    all_label_ids = torch.tensor(all_label_ids, dtype=torch.long)

    return all_token_ids, all_label_ids

然后进行长链切块 + shift：
- reshape 成 (num_chunks, seq_len + 1)
- input_ids = chunk[:, :-1]
- labels    = chunk[:, 1:]

In [ ]:
def chunk_and_shift_packed_stream(
    all_token_ids: torch.Tensor,
    all_label_ids: torch.Tensor,
    seq_length: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    将 packed 长链切成固定块，并做 causal LM shift。
    返回:
      final_input_ids: [num_chunks, seq_length]
      final_labels:    [num_chunks, seq_length]
    """
    chunk_size = seq_length + 1
    num_chunks = len(all_token_ids) // chunk_size

    if num_chunks == 0:
        raise ValueError(
            f"数据太小：总 token 数 {len(all_token_ids)} < chunk_size {chunk_size}"
        )

    input_chunks = all_token_ids[: num_chunks * chunk_size].view(num_chunks, chunk_size)
    label_chunks = all_label_ids[: num_chunks * chunk_size].view(num_chunks, chunk_size)

    final_input_ids = input_chunks[:, :-1].contiguous()
    final_labels = label_chunks[:, 1:].contiguous()

    return final_input_ids, final_labels

最后过滤死块：

In [ ]:
def filter_dead_chunks(
    final_input_ids: torch.Tensor,
    final_labels: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, int]:
    """
    过滤 labels 全为 -100 的死块。
    返回:
      filtered_input_ids, filtered_labels, dropped_count
    """
    valid_chunk_mask = (final_labels != -100).any(dim=1)

    dropped_count = int((~valid_chunk_mask).sum().item())

    filtered_input_ids = final_input_ids[valid_chunk_mask]
    filtered_labels = final_labels[valid_chunk_mask]

    return filtered_input_ids, filtered_labels, dropped_count

### 7.2.5 `InstructionDataset`组装

In [ ]:
class InstructionDataset(Dataset):
    def __init__(
        self,
        tokenizer,
        dataset_path: str,
        seq_length: int,
        shuffle: bool = True,
        apply_masking: bool = True,
        seed: int = 42,
    ):
        self.tokenizer = tokenizer
        self.seq_length = seq_length
        self.apply_masking = apply_masking

        docs = load_instruction_docs(dataset_path)

        all_token_ids, all_label_ids = build_packed_token_stream(
            docs=docs,
            tokenizer=tokenizer,
            shuffle_docs=shuffle,
            apply_masking=apply_masking,
            seed=seed,
        )

        final_input_ids, final_labels = chunk_and_shift_packed_stream(
            all_token_ids=all_token_ids,
            all_label_ids=all_label_ids,
            seq_length=seq_length,
        )

        final_input_ids, final_labels, dropped_count = filter_dead_chunks(
            final_input_ids=final_input_ids,
            final_labels=final_labels,
        )

        self.final_input_ids = final_input_ids
        self.final_labels = final_labels

        print(f"Loaded {len(docs)} docs")
        print(f"Packed into {len(self.final_input_ids)} valid chunks")
        print(f"Dropped dead chunks: {dropped_count}")

    def __len__(self):
        return len(self.final_input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.final_input_ids[idx].clone(),
            "labels": self.final_labels[idx].clone(),
        }

### 7.2.6 dataset smoke test

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_path"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_ds_preview = InstructionDataset(
    tokenizer=tokenizer,
    dataset_path=CFG["train_path"],
    seq_length=CFG["max_seq_len"],
    shuffle=CFG["shuffle_docs"],
    apply_masking=CFG["apply_masking"],
    seed=CFG["seed"],
)

sample = train_ds_preview[0]
print(sample["input_ids"].shape, sample["labels"].shape)
print("有效监督 token 数:", (sample["labels"] != -100).sum().item())

decoded_preview = tokenizer.decode(sample["input_ids"][:300], skip_special_tokens=False)
print(decoded_preview[:1500])

### 7.2.7 训练基础设施
`compute_entropy`记得额外导入进来。

In [ ]:
def init_instruction_sft_experiment(cfg: Dict):
    wandb.init(
        project=cfg["wandb_project"],
        name=cfg["wandb_run_name"],
        config=cfg,
    )

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_path"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        cfg["model_path"],
        torch_dtype=dtype,
        attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
    ).to(DEVICE)

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    return tokenizer, model

然后构建 train/eval 的 dataloader：

In [ ]:
def build_instruction_dataloaders(tokenizer, cfg: Dict):
    train_ds = InstructionDataset(
        tokenizer=tokenizer,
        dataset_path=cfg["train_path"],
        seq_length=cfg["max_seq_len"],
        shuffle=cfg["shuffle_docs"],
        apply_masking=cfg["apply_masking"],
        seed=cfg["seed"],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["micro_batch_size"],
        shuffle=True,
    )

    eval_loader = None
    if cfg["eval_path"] is not None:
        eval_ds = InstructionDataset(
            tokenizer=tokenizer,
            dataset_path=cfg["eval_path"],
            seq_length=cfg["max_seq_len"],
            shuffle=False,
            apply_masking=cfg["apply_masking"],
            seed=cfg["seed"],
        )
        eval_loader = DataLoader(
            eval_ds,
            batch_size=cfg["micro_batch_size"],
            shuffle=False,
        )

    return train_loader, eval_loader

def build_optimizer_and_scheduler(model, train_loader, cfg: Dict):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=0.0,
    )

    grad_accum_steps = max(1, cfg["batch_size"] // cfg["micro_batch_size"])
    total_steps = (len(train_loader) // grad_accum_steps) * cfg["epochs"]

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(cfg["warmup_ratio"] * total_steps),
        num_training_steps=total_steps,
    )

    return optimizer, scheduler, grad_accum_steps, total_steps

验证函数如下：

In [ ]:
@torch.no_grad()
def run_evaluation(model, eval_loader, max_batches: int = 50):
    model.eval()

    total_loss = 0.0
    total_entropy = 0.0
    count = 0

    for i, batch in enumerate(eval_loader):
        if i >= max_batches:
            break

        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids=input_ids).logits

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100,
        )
        total_loss += loss.item()

        ent_all = compute_entropy(logits)
        active_mask = (labels != -100)
        if active_mask.any():
            total_entropy += ent_all[active_mask].mean().item()

        count += 1

    return {
        "eval/loss": total_loss / max(1, count),
        "eval/response_entropy": total_entropy / max(1, count),
    }

### 7.2.8 训练循环
首先是单个 micro-batch 前向与统计
- 前向是去计算 CE loss；
- 统计的是 active token entropy

In [ ]:
def compute_instruction_sft_microbatch(model, batch: Dict, grad_accum_steps: int):
    input_ids = batch["input_ids"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    logits = model(input_ids=input_ids).logits

    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
    )

    scaled_loss = loss / grad_accum_steps

    with torch.no_grad():
        ent_all = compute_entropy(logits)
        active_mask = (labels != -100)

        response_entropy = ent_all[active_mask].mean().item() if active_mask.any() else 0.0
        active_tokens = active_mask.sum().item()

    return {
        "loss": loss,
        "scaled_loss": scaled_loss,
        "response_entropy": response_entropy,
        "active_tokens": active_tokens,
    }

经典若干 micro-batch 累积之后进行一次 optimizer update：

In [ ]:
def train_one_optimizer_step(
    model,
    optimizer,
    scheduler,
    train_loader_iter,
    grad_accum_steps: int,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    accumulated_loss = 0.0
    accumulated_entropy = 0.0
    actual_micro_steps = 0

    for _ in range(grad_accum_steps):
        try:
            batch = next(train_loader_iter)
        except StopIteration:
            break

        outputs = compute_instruction_sft_microbatch(
            model=model,
            batch=batch,
            grad_accum_steps=grad_accum_steps,
        )

        outputs["scaled_loss"].backward()

        accumulated_loss += outputs["loss"].item()
        accumulated_entropy += outputs["response_entropy"]
        actual_micro_steps += 1

    if actual_micro_steps == 0:
        return None

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    return {
        "train/loss": accumulated_loss / actual_micro_steps,
        "train/response_entropy": accumulated_entropy / actual_micro_steps,
        "train/lr": optimizer.param_groups[0]["lr"],
    }

def run_instruction_sft(cfg: Dict):
    os.makedirs(cfg["output_dir"], exist_ok=True)

    tokenizer, model = init_instruction_sft_experiment(cfg)
    train_loader, eval_loader = build_instruction_dataloaders(tokenizer, cfg)
    optimizer, scheduler, grad_accum_steps, total_steps = build_optimizer_and_scheduler(
        model, train_loader, cfg
    )

    print(f"训练总更新步数: {total_steps}")
    print(f"梯度累积步数: {grad_accum_steps}")

    global_step = 0
    progress_bar = tqdm(total=total_steps, desc="Instruction SFT")

    for epoch in range(cfg["epochs"]):
        train_loader_iter = iter(train_loader)

        while True:
            step_metrics = train_one_optimizer_step(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                train_loader_iter=train_loader_iter,
                grad_accum_steps=grad_accum_steps,
            )

            if step_metrics is None:
                break

            global_step += 1
            step_metrics["train/epoch"] = epoch + global_step / max(1, total_steps)

            # 周期性验证
            if eval_loader is not None and global_step % cfg["eval_every_steps"] == 0:
                eval_metrics = run_evaluation(model, eval_loader)
                step_metrics.update(eval_metrics)
                model.train()

            # 周期性保存
            if global_step % cfg["save_every_steps"] == 0:
                ckpt_path = os.path.join(cfg["output_dir"], f"checkpoint-{global_step}")
                os.makedirs(ckpt_path, exist_ok=True)
                model.save_pretrained(ckpt_path)
                tokenizer.save_pretrained(ckpt_path)

            wandb.log(step_metrics, step=global_step)

            progress_bar.set_postfix({
                "loss": f"{step_metrics['train/loss']:.4f}",
                "lr": f"{step_metrics['train/lr']:.2e}",
            })
            progress_bar.update(1)

    model.save_pretrained(cfg["output_dir"])
    tokenizer.save_pretrained(cfg["output_dir"])
    wandb.finish()

    return model, tokenizer

In [ ]:
model, tokenizer = run_instruction_sft(CFG)
# 这一节不涉及新的训练理论，所以可以比较直接的归类成3层：
# 数据层：load_instruction_docs、tokenize_instruction_example、build_packed_token_stream
# packing 层：chunk_and_shift_packed_stream、filter_dead_chunks
# 训练层：compute_instruction_sft_microbatch、train_one_optimizer_step、run_instruction_sft